# §13.8.5 — 층별 부품 제거로 측정하는 기여

> 딥러닝 교재 · 3부 13장 8절 5항 (🐍)
> 선행: §13.8.1(블록 수식화) · §13.8.2(토큰/채널 혼합) · §9.6(얕은 경로 앙상블)

## 이 노트북이 답하는 질문

1. **층×부품(어텐션/MLP)의 기여는 균일한가?** 하나씩 항등으로 바꿔 잰다.
2. **어텐션 전부 제거와 MLP 전부 제거는 어떻게 다른가?** 두 혼합 축의 비대칭.
3. **부품 제거가 남은 부품의 어텐션 패턴에 어떤 흔적을 남기는가?** 회로의 부검.

**예상 실행 시간** CPU 약 2분 30초 (`FAST = True`이면 약 80초).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 과제와 모델 — 회로가 읽히는 복제 언어

열의 앞 절반은 서로 다른 무작위 토큰, 뒤 절반은 그 복제다. 뒤 절반의 예측을 푸는
교과서적 회로는 **층 협업의 유도 회로**다 — 앞 층의 이전-토큰 헤드가 각 토큰에
'내 앞 토큰' 정보를 실어 주면, 뒤 층의 유도 헤드가 그것을 키로 조회한다. 정말 그
회로가 학습되는지, 아니면 다른 지름길이 있는지를 부품 수술로 부검한다. 잔차
덕분에 부품을 항등으로 바꿔치기하는 수술(`skip`)이 가능하다.

모델은 공용 미니 트랜스포머(3층, $h=4$, $d=48$)다. 손실은 복제 구간에서만 잰다.

In [ ]:
# ── 공용 미니 트랜스포머 (NumPy, 완전한 순전파+역전파) ──────────
# 구조: 임베딩 → L × [Pre-LN 블록 (MHA + MLP, 잔차)] → LN → 판독
# 위치 부호화: 'learned' | 'sin' | 'rope' | 'alibi' | 'none'

def make_config(V, d=48, L=2, h=4, T_max=64, pe='learned', causal=True, seed=0):
    dh = d // h
    rn = np.random.default_rng(seed)
    p = {}
    p['emb'] = rn.standard_normal((V, d)) * 0.5 / np.sqrt(d)
    if pe == 'learned':
        p['pos'] = rn.standard_normal((T_max, d)) * 0.5 / np.sqrt(d)
    for l in range(L):
        s = f'l{l}_'
        for nm in ['wq', 'wk', 'wv', 'wo']:
            p[s + nm] = rn.standard_normal((d, d)) / np.sqrt(d)
        p[s + 'ln1g'] = np.ones(d); p[s + 'ln1b'] = np.zeros(d)
        p[s + 'w1'] = rn.standard_normal((d, 4 * d)) / np.sqrt(d)
        p[s + 'b1'] = np.zeros(4 * d)
        p[s + 'w2'] = rn.standard_normal((4 * d, d)) / np.sqrt(4 * d)
        p[s + 'b2'] = np.zeros(d)
        p[s + 'ln2g'] = np.ones(d); p[s + 'ln2b'] = np.zeros(d)
    p['lnfg'] = np.ones(d); p['lnfb'] = np.zeros(d)
    p['out'] = rn.standard_normal((d, V)) / np.sqrt(d)
    cfg = dict(V=V, d=d, L=L, h=h, dh=dh, T_max=T_max, pe=pe, causal=causal)
    return p, cfg

def _sin_pe(T, d):
    pos = np.arange(T)[:, None]
    l2 = np.arange(0, d, 2)[None, :]
    ang = pos / (10000.0 ** (l2 / d))
    pe = np.zeros((T, d))
    pe[:, 0::2] = np.sin(ang); pe[:, 1::2] = np.cos(ang)
    return pe

def _rope_angles(T, dh, scale=1.0):
    pos = np.arange(T)[:, None] * scale
    l2 = np.arange(0, dh, 2)[None, :]
    return pos / (10000.0 ** (l2 / dh))          # (T, dh/2)

def _rope_apply(x, ang, inverse=False):
    # x: (B,h,T,dh) — 짝수/홀수 쌍을 각도 ang(T,dh/2)만큼 회전
    c, s = np.cos(ang), np.sin(ang)
    if inverse:
        s = -s
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return np.stack([x1 * c - x2 * s, x1 * s + x2 * c], axis=-1).reshape(x.shape)

def _alibi_slopes(h):
    return np.array([2.0 ** (-8.0 * (i + 1) / h) for i in range(h)])

def _ln_f(x, g, b):
    mu = x.mean(-1, keepdims=True)
    xc = x - mu
    var = (xc ** 2).mean(-1, keepdims=True)
    inv = 1.0 / np.sqrt(var + 1e-5)
    xh = xc * inv
    return xh * g + b, (xh, inv)

def _ln_b(dy, cache, g):
    xh, inv = cache
    dxh = dy * g
    dg = (dy * xh).sum(axis=tuple(range(dy.ndim - 1)))
    db = dy.sum(axis=tuple(range(dy.ndim - 1)))
    dx = inv * (dxh - dxh.mean(-1, keepdims=True) - xh * (dxh * xh).mean(-1, keepdims=True))
    return dx, dg, db

def forward(p, cfg, idx, targets=None, rope_scale=1.0, head_mask=None,
            skip=None, want_attn=False, kv_keep=None):
    """idx:(B,T) 정수. targets:(B,T) 또는 None.
    head_mask:(L,h) 0/1, skip: {'attn':set(l), 'mlp':set(l)},
    kv_keep:(T,) bool — 열 s의 키·값 사용 여부(캐시 축출 흉내)."""
    B, T = idx.shape
    d, L, h, dh = cfg['d'], cfg['L'], cfg['h'], cfg['dh']
    x = p['emb'][idx]                              # (B,T,d)
    if cfg['pe'] == 'learned':
        x = x + p['pos'][:T]
    elif cfg['pe'] == 'sin':
        x = x + _sin_pe(T, d)
    cache = {'idx': idx, 'T': T, 'B': B, 'xs': [], 'attn': []}
    ang = _rope_angles(T, dh, rope_scale) if cfg['pe'] == 'rope' else None
    if cfg['causal']:
        nmask = np.triu(np.full((T, T), -np.inf), k=1)
    else:
        nmask = np.zeros((T, T))
    if kv_keep is not None:
        nmask = nmask.copy()
        nmask[:, ~kv_keep] = -np.inf
    if cfg['pe'] == 'alibi':
        sl = _alibi_slopes(h)
        dist = np.maximum(np.arange(T)[:, None] - np.arange(T)[None, :], 0)
        abias = -sl[:, None, None] * dist[None]    # (h,T,T)
    else:
        abias = np.zeros((1, T, T))
    skip = skip or {'attn': set(), 'mlp': set()}
    attns = []
    def split(z):
        return z.reshape(B, T, h, dh).transpose(0, 2, 1, 3)       # (B,h,T,dh)
    for l in range(L):
        s = f'l{l}_'
        c = {}
        if l not in skip['attn']:
            # ── MHA 가지 ──
            h1, c['ln1'] = _ln_f(x, p[s + 'ln1g'], p[s + 'ln1b'])
            c['h1'] = h1
            q = split(h1 @ p[s + 'wq']); k = split(h1 @ p[s + 'wk']); v = split(h1 @ p[s + 'wv'])
            if cfg['pe'] == 'rope':
                q = _rope_apply(q, ang); k = _rope_apply(k, ang)
            e = np.einsum('bhtd,bhsd->bhts', q, k) / np.sqrt(dh) + nmask + abias[None]
            e -= e.max(-1, keepdims=True)
            a = np.exp(e); a /= a.sum(-1, keepdims=True)
            if head_mask is not None:
                hm = head_mask[l][None, :, None, None]
            else:
                hm = 1.0
            av = np.einsum('bhts,bhsd->bhtd', a, v) * hm
            avm = av.transpose(0, 2, 1, 3).reshape(B, T, d)
            x = x + avm @ p[s + 'wo']
            c.update(q=q, k=k, v=v, a=a, avm=avm, hm=hm)
            attns.append(a)
        else:
            attns.append(None)
        if l not in skip['mlp']:
            # ── MLP 가지 ──
            h2, c['ln2'] = _ln_f(x, p[s + 'ln2g'], p[s + 'ln2b'])
            z1 = h2 @ p[s + 'w1'] + p[s + 'b1']
            r = np.maximum(z1, 0.0)               # ReLU (역전파 단순화)
            x = x + r @ p[s + 'w2'] + p[s + 'b2']
            c['mlp'] = (h2, z1, r)
        else:
            c['mlp'] = None
        cache[s] = c
    hf, cache['lnf'] = _ln_f(x, p['lnfg'], p['lnfb'])
    cache['hf'] = hf
    logits = hf @ p['out']
    cache['logits'] = logits
    out = {'logits': logits}
    if want_attn:
        out['attn'] = attns
    if targets is not None:
        valid = targets >= 0                      # -1 = 손실에서 제외
        tsafe = np.maximum(targets, 0)
        z = logits - logits.max(-1, keepdims=True)
        lse = np.log(np.exp(z).sum(-1))
        ll = z[np.arange(B)[:, None], np.arange(T)[None, :], tsafe] - lse
        out['loss'] = -(ll * valid).sum() / max(valid.sum(), 1)
        P = np.exp(z); P /= P.sum(-1, keepdims=True)
        cache['P'] = P; cache['targets'] = tsafe; cache['valid'] = valid
    out['cache'] = cache
    return out

def backward(p, cfg, cache, rope_scale=1.0):
    B, T = cache['B'], cache['T']
    d, L, h, dh = cfg['d'], cfg['L'], cfg['h'], cfg['dh']
    g = {k: np.zeros_like(v) for k, v in p.items()}
    P, targets, valid = cache['P'], cache['targets'], cache['valid']
    dlogits = P.copy()
    dlogits[np.arange(B)[:, None], np.arange(T)[None, :], targets] -= 1.0
    dlogits *= valid[:, :, None]
    dlogits /= max(valid.sum(), 1)
    hf = cache['hf']
    g['out'] = np.einsum('btd,btv->dv', hf, dlogits)
    dhf = dlogits @ p['out'].T
    dx, g['lnfg'], g['lnfb'] = _ln_b(dhf, cache['lnf'], p['lnfg'])
    ang = _rope_angles(T, dh, rope_scale) if cfg['pe'] == 'rope' else None
    for l in range(L - 1, -1, -1):
        s = f'l{l}_'
        c = cache[s]
        if c['mlp'] is not None:
            h2, z1, r = c['mlp']
            dmlp = dx                                   # 잔차: 가지로 흘러드는 기울기
            g[s + 'w2'] += np.einsum('btf,btd->fd', r, dmlp)
            g[s + 'b2'] += dmlp.sum((0, 1))
            dr = dmlp @ p[s + 'w2'].T
            dz1 = dr * (z1 > 0)
            g[s + 'w1'] += np.einsum('btd,btf->df', h2, dz1)
            g[s + 'b1'] += dz1.sum((0, 1))
            dh2 = dz1 @ p[s + 'w1'].T
            dxi, dg2, db2 = _ln_b(dh2, c['ln2'], p[s + 'ln2g'])
            g[s + 'ln2g'] += dg2; g[s + 'ln2b'] += db2
            dx = dx + dxi
        if 'a' not in c:
            continue
        # MHA 가지
        dattn_out = dx
        g[s + 'wo'] += np.einsum('btd,bte->de', c['avm'], dattn_out)
        davm = dattn_out @ p[s + 'wo'].T
        dav = davm.reshape(B, T, h, dh).transpose(0, 2, 1, 3) * c['hm']
        a, q, k, v = c['a'], c['q'], c['k'], c['v']
        da = np.einsum('bhtd,bhsd->bhts', dav, v)
        dv = np.einsum('bhts,bhtd->bhsd', a, dav)
        de = a * (da - (a * da).sum(-1, keepdims=True))
        dq = np.einsum('bhts,bhsd->bhtd', de, k) / np.sqrt(dh)
        dk = np.einsum('bhts,bhtd->bhsd', de, q) / np.sqrt(dh)
        if cfg['pe'] == 'rope':
            dq = _rope_apply(dq, ang, inverse=True)
            dk = _rope_apply(dk, ang, inverse=True)
        def merge(z):
            return z.transpose(0, 2, 1, 3).reshape(B, T, d)
        dq, dk, dv = merge(dq), merge(dk), merge(dv)
        h1 = c['h1']
        g[s + 'wq'] += np.einsum('btd,bte->de', h1, dq)
        g[s + 'wk'] += np.einsum('btd,bte->de', h1, dk)
        g[s + 'wv'] += np.einsum('btd,bte->de', h1, dv)
        dh1 = dq @ p[s + 'wq'].T + dk @ p[s + 'wk'].T + dv @ p[s + 'wv'].T
        dxi, dg1, db1 = _ln_b(dh1, c['ln1'], p[s + 'ln1g'])
        g[s + 'ln1g'] += dg1; g[s + 'ln1b'] += db1
        dx = dx + dxi
    if cfg['pe'] == 'learned':
        g['pos'][:T] += dx.sum(0)
    np.add.at(g['emb'], cache['idx'], dx)
    return g

def adam_init(p):
    return {k: np.zeros_like(v) for k, v in p.items()}, {k: np.zeros_like(v) for k, v in p.items()}

def adam_step(p, g, m, v, t, lr=3e-3):
    for k in p:
        m[k] = 0.9 * m[k] + 0.1 * g[k]
        v[k] = 0.999 * v[k] + 0.001 * g[k] ** 2
        p[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)

def train_lm(p, cfg, sample_batch, steps, lr=3e-3, log_every=0, rope_scale=1.0):
    m, v = adam_init(p)
    hist = []
    for t in range(1, steps + 1):
        idx, tgt = sample_batch()
        out = forward(p, cfg, idx, targets=tgt, rope_scale=rope_scale)
        g = backward(p, cfg, out['cache'], rope_scale=rope_scale)
        adam_step(p, g, m, v, t, lr)
        hist.append(out['loss'])
        if log_every and t % log_every == 0:
            print(f"  step {t}: loss {np.mean(hist[-log_every:]):.3f}")
    return hist

In [ ]:
V = 64
T_SEQ = 48
HALF = T_SEQ // 2
L_DEPTH = 3

def dup_batch(B, rn):
    seq = np.array([rn.choice(np.arange(2, V), size=HALF, replace=False) for _ in range(B)])
    idx = np.concatenate([seq, seq], axis=1)
    idx[:, 0] = 1
    tgt = np.full((B, T_SEQ), -1)
    tgt[:, HALF - 1:T_SEQ - 1] = idx[:, HALF:]
    return idx, tgt

p, cfg = make_config(V=V, d=48, L=L_DEPTH, h=4, T_max=T_SEQ, pe='learned', seed=3)
rn = np.random.default_rng(200)
STEPS = 250 if FAST else 500
train_lm(p, cfg, lambda: dup_batch(32, rn), steps=STEPS, lr=3e-3)
idx_ev, tgt_ev = dup_batch(256, np.random.default_rng(31))
base = forward(p, cfg, idx_ev, targets=tgt_ev)['loss']
print(f"학습 후 복제 구간 손실 {base:.3f} (균등 분포 {np.log(V-2):.2f})")

---
## 2. 층×부품 제거 지도

In [ ]:
def ablate(skip_attn=(), skip_mlp=()):
    return forward(p, cfg, idx_ev, targets=tgt_ev,
                   skip={'attn': set(skip_attn), 'mlp': set(skip_mlp)})['loss']

grid = np.zeros((L_DEPTH, 2))
for l in range(L_DEPTH):
    grid[l, 0] = ablate(skip_attn=[l]) - base
    grid[l, 1] = ablate(skip_mlp=[l]) - base
    print(f"층 {l}:  어텐션 제거 Δ{grid[l,0]:+.3f}   MLP 제거 Δ{grid[l,1]:+.3f}")

all_attn = ablate(skip_attn=range(L_DEPTH))
all_mlp = ablate(skip_mlp=range(L_DEPTH))
print(f"어텐션 전부 제거: {all_attn:.2f}  |  MLP 전부 제거: {all_mlp:.2f}  |  원본 {base:.3f}")

---
## 3. 누적 제거 — 잔차 배선의 여유분

In [ ]:
parts = [('attn', l) for l in range(L_DEPTH)] + [('mlp', l) for l in range(L_DEPTH)]
removed_a, removed_m = set(), set()
cum = [(0, base)]
for k in range(1, len(parts) + 1):
    cand = []
    for kind, l in parts:
        if (kind == 'attn' and l in removed_a) or (kind == 'mlp' and l in removed_m):
            continue
        sa = removed_a | ({l} if kind == 'attn' else set())
        sm = removed_m | ({l} if kind == 'mlp' else set())
        cand.append((ablate(skip_attn=sa, skip_mlp=sm), kind, l))
    lo, kind, l = min(cand)
    (removed_a if kind == 'attn' else removed_m).add(l)
    cum.append((k, lo))
    print(f"{k}개 제거 (+{kind}{l}): 손실 {lo:.3f}")
cum = np.array(cum)

---
## 4. 부검 — 지도가 밝힌 회로의 정체

(a)의 지도는 뜻밖의 사실을 말한다. **0층 어텐션 하나가 거의 전부이고, 1·2층 어텐션은
지워도 무방하다.** 유도 회로(이전-토큰 헤드 → 유도 헤드의 2층 협업)를 기대했다면
빗나간 것이다. 정체를 확인하러 0층 헤드들의 어텐션을 **상대 오프셋의 함수**로 편다.
이 과제의 정답 조회 자리는 언제나 $s^*=t-{\rm half}+1$, 즉 상대 오프셋 $-23$이다 —
복제 오프셋이 고정이므로, 학습된 위치 부호화만으로 "23칸 앞을 볼 것"이라는 회로가
한 층에서 조립된다. §13.4.6에서 외삽을 망치던 바로 그 위치 지름길이, 여기서는
모델이 실제로 배운 회로다.

In [ ]:
out0 = forward(p, cfg, idx_ev[:64], targets=tgt_ev[:64], want_attn=True)
A0 = out0['attn'][0]                                   # (B,h,T,T) — 0층
ts = np.arange(HALF, T_SEQ)
offsets = np.arange(-HALF - 4, 1)
prof = np.zeros((4, len(offsets)))
for oi, r in enumerate(offsets):
    ss = ts + r
    ok = ss >= 0
    prof[:, oi] = A0[:, :, ts[ok], ss[ok]].mean(axis=(0, 2))
print(f"0층 헤드별 최대 질량 오프셋: {offsets[prof.argmax(1)]}  (정답 조회 오프셋 = {-(HALF-1)})")

---
## 5. 교재 그림 — fig_13_8_5

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 층×부품 지도
ax = axes[0]
imv = ax.imshow(grid.T, cmap='viridis', aspect='auto')
plt.colorbar(imv, ax=ax, fraction=0.046)
ax.set_xticks(range(L_DEPTH)); ax.set_xticklabels([f'{l}' for l in range(L_DEPTH)])
ax.set_yticks([0, 1]); ax.set_yticklabels([lab('어텐션', 'attn'), 'MLP'])
for l in range(L_DEPTH):
    for j in range(2):
        ax.text(l, j, f'{grid[l,j]:+.2f}', ha='center', va='center',
                color='w' if grid[l, j] > grid.max()*0.5 else 'k', fontsize=9)
ax.grid(False)
ax.set_xlabel(lab('층', 'layer'))
ax.set_title(lab('(a) 부품 하나 제거 시 손실 증가', '(a) ablation map'), fontsize=10)

# (b) 전부 제거의 비대칭
ax = axes[1]
xs = np.arange(3)
vals = [base, all_mlp, all_attn]
cols = [CB[5], CB[1], CB[4]]
ax.bar(xs, vals, 0.55, color=cols)
ax.axhline(np.log(V - 2), color='k', lw=0.8, ls=':')
ax.text(0.0, np.log(V - 2) + 0.1, lab('균등 분포', 'uniform'), fontsize=8)
ax.set_xticks(xs)
ax.set_xticklabels([lab('원본', 'full'), lab('MLP 전부 제거', 'no MLP'),
                    lab('어텐션 전부 제거', 'no attn')], fontsize=9)
ax.set_ylabel(lab('복제 구간 손실', 'loss'))
ax.set_title(lab('(b) 토큰 혼합 없이는 단어 주머니뿐', '(b) asymmetry'), fontsize=10)

# (c) 누적 제거
ax = axes[2]
ax.plot(cum[:, 0], cum[:, 1], 'o-', color=CB[5], ms=5)
ax.axhline(np.log(V - 2), color='k', lw=0.8, ls=':')
ax.set_xlabel(lab('제거한 부품 수 (탐욕 순서)', 'parts removed'))
ax.set_ylabel(lab('손실', 'loss'))
ax.set_title(lab('(c) 여유분이 소진되는 순간', '(c) cumulative removal'), fontsize=10)

# (d) 0층 헤드의 오프셋 프로파일
ax = axes[3]
for hh in range(4):
    ax.plot(offsets, prof[hh], '-', color=CB[hh + 1], lw=1.3, label=f'H{hh}')
ax.axvline(-(HALF - 1), color='k', lw=0.8, ls=':')
ax.text(-(HALF - 1) + 0.5, prof.max() * 0.9, lab('정답 조회 오프셋', 'correct offset'), fontsize=8)
ax.set_xlabel(lab('상대 오프셋 $s-t$', 'relative offset'))
ax.set_ylabel(lab('0층 평균 어텐션 질량', 'L0 attn mass'))
ax.set_title(lab('(d) 부검 — 고정 오프셋 복사 회로였다', '(d) circuit autopsy'), fontsize=10)
ax.legend(fontsize=8)

save_book_fig(fig, 'fig_13_8_5')
plt.show()

> ### 읽는 법
>
> (a) 기여는 층과 부품에 걸쳐 극단적으로 불균일하다 — 0층 어텐션이 홀로 지배적이고
> (그 이유는 (d)의 부검이 밝힌다), 이 과제에서 MLP들의 한계 기여는 작다.
> (b) 비대칭의 핵심. 어텐션을 전부 지우면 토큰 사이의 혼합이 사라져 모델은 위치별
> 통계 이상을 볼 수 없는 단어 주머니가 되고(균등 분포 근처), MLP를 전부 지워도
> 토큰 혼합은 남으므로 열화가 훨씬 완만하다 — §13.8.2의 두 혼합 축이 대체 불가능한
> 서로 다른 축이라는 실측이다.
> (c) 잔차 배선 덕에 기여 낮은 부품들은 여럿 지워도 완만하다가, 회로의 필수 부품이
> 잘리는 순간 절벽이 온다(§9.6의 얕은 경로 앙상블 그림).
> (d) 부검의 반전. 우리는 2층짜리 유도 회로를 기대했지만, 지도는 0층 어텐션 하나가
> 전부라고 말했고, 오프셋 프로파일이 그 정체를 확정한다 — 헤드들이 정확히 정답
> 오프셋 $-23$에 질량을 몰아 두는 **고정 오프셋 복사 회로**다. 복제 오프셋이 상수인
> 과제에서는 이 지름길이 유도 회로보다 싸다(§13.4.6에서 외삽을 망치던 바로 그
> 지름길이다). 제거 실험의 미덕이 여기 있다 — 우리가 기대한 회로가 아니라 모델이
> 실제로 배운 회로를 보여 준다.

---
## 6. 자기 점검

1. 재출현 간격이 무작위인 유도 문법(§13.4.6의 데이터)으로 바꿔 다시 학습하면 (a)의 지도가 어떻게 달라지는가? 고정 오프셋 지름길이 막히면 몇 개 층의 어텐션이 필수가 되는지 확인하라.
2. 이 과제 대신 마르코프 연쇄(§13.5.4의 데이터)로 학습하면 (b)의 비대칭이 어떻게 변하겠는가? 실행해 보라 — 채널 혼합이 담당할 통계가 늘어난다.
3. (c)의 탐욕 순서와 (a)의 한계 기여 순서가 갈리는 지점이 있는가? 부품 간 중복의 증거를 찾아보라.
4. 제거 대신 **재학습**(제거 후 짧게 미세조정)을 하면 (b)의 두 값이 얼마나 회복되는가? 구조의 기여와 파라미터의 기여를 구분하라.

## 7. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `L_DEPTH` | 1절 | 3 | 여유분(잉여 경로)의 양 |
| `STEPS` | 1절 | 500 | 수렴도와 회로의 선명도 |
| 과제 | 1절 | 복제 | 마르코프로 바꾸면 MLP 기여 ↑ |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")